In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

In [2]:
leagues = ["bundesliga","la_liga","ligue_1","premier_league","serie_a"]
years = ["2020","2021","2022","2023","2024"]

full_data = []

In [3]:
matches_list = []
teams_list = []
players_list = []
for league in leagues:
    for year in years:
        df = pd.read_json(f"../data/raw/understat/{league}/matches_{year}.json")
        df['league'] = league
        df['year'] = year
        matches_list.append(df)

        df2=pd.read_json(f"../data/raw/understat/{league}/players_{year}.json")
        df2['league'] = league
        df2['year'] = year
        players_list.append(df2)

        with open(f"../data/raw/understat/{league}/teams_{year}.json") as f:
            raw = json.load(f)
        for team_id,team_data in raw.items():
            df1 = pd.DataFrame(team_data['history'])
            df1['team_id'] = team_id
            df1['team_name'] = team_data['title']
            df1['league'] = league
            df1['year'] = year
            teams_list.append(df1)

matches_df = pd.concat(matches_list, ignore_index = True)
teams_df = pd.concat(teams_list, ignore_index = True)
players_df = pd.concat(players_list, ignore_index = True)

print("matches_df: ", matches_df.shape)
print("teams_df: ", teams_df.shape)
print("players_df: ", players_df.shape)

matches_df:  (8982, 10)
teams_df:  (17964, 23)
players_df:  (13964, 20)


In [4]:
matches_df.head()
matches_df.dtypes

id                   int64
isResult              bool
h                   object
a                   object
goals               object
xG                  object
datetime    datetime64[us]
forecast            object
league                 str
year                   str
dtype: object

In [5]:
teams_df.head()
teams_df.dtypes

h_a                 str
xG              float64
xGA             float64
npxG            float64
npxGA           float64
ppda             object
ppda_allowed     object
deep              int64
deep_allowed      int64
scored            int64
missed            int64
xpts            float64
result              str
date                str
wins              int64
draws             int64
loses             int64
pts               int64
npxGD           float64
team_id             str
team_name           str
league              str
year                str
dtype: object

In [6]:
players_df.head()
players_df.dtypes

id                int64
player_name         str
games             int64
time              int64
goals             int64
xG              float64
assists           int64
xA              float64
shots             int64
key_passes        int64
yellow_cards      int64
red_cards         int64
position            str
team_title          str
npg               int64
npxG            float64
xGChain         float64
xGBuildup       float64
league              str
year                str
dtype: object

In [7]:
matches_df.head()

,id,isResult,h,a,goals,xG,datetime,forecast,league,year
0,14173,True,"{'id': '117', 'title': 'Bayern Munich', 'short...","{'id': '124', 'title': 'Schalke 04', 'short_ti...","{'h': '8', 'a': '0'}","{'h': '4.62088', 'a': '0.14975'}",2020-09-18 18:30:00,"{'w': '0.9948', 'd': '0.0046', 'l': '0.0006'}",bundesliga,2020
1,14174,True,"{'id': '132', 'title': 'Eintracht Frankfurt', ...","{'id': '262', 'title': 'Arminia Bielefeld', 's...","{'h': '1', 'a': '1'}","{'h': '2.46928', 'a': '0.618787'}",2020-09-19 13:30:00,"{'w': '0.8454', 'd': '0.1158', 'l': '0.0388'}",bundesliga,2020
2,14175,True,"{'id': '240', 'title': 'Union Berlin', 'short_...","{'id': '121', 'title': 'Augsburg', 'short_titl...","{'h': '1', 'a': '3'}","{'h': '1.04515', 'a': '1.42046'}",2020-09-19 13:30:00,"{'w': '0.243', 'd': '0.2882', 'l': '0.4688'}",bundesliga,2020
3,14176,True,"{'id': '134', 'title': 'FC Cologne', 'short_ti...","{'id': '120', 'title': 'Hoffenheim', 'short_ti...","{'h': '2', 'a': '3'}","{'h': '2.40801', 'a': '2.9032'}",2020-09-19 13:30:00,"{'w': '0.2453', 'd': '0.253', 'l': '0.5017'}",bundesliga,2020
4,14177,True,"{'id': '123', 'title': 'Werder Bremen', 'short...","{'id': '122', 'title': 'Hertha Berlin', 'short...","{'h': '1', 'a': '4'}","{'h': '0.495892', 'a': '2.05406'}",2020-09-19 13:30:00,"{'w': '0.0616', 'd': '0.1641', 'l': '0.7743'}",bundesliga,2020


In [8]:
h_df = pd.json_normalize(matches_df['h'])
h_df.rename(columns = {"title":"home_team","short_title":"home_short"},inplace = True)
h_df.head()

,id,home_team,home_short
0,117,Bayern Munich,BAY
1,132,Eintracht Frankfurt,EIN
2,240,Union Berlin,UNI
3,134,FC Cologne,COL
4,123,Werder Bremen,WER


In [9]:
a_df = pd.json_normalize(matches_df['a'])
a_df.rename(columns = {"title":"away_team","short_title":"away_short"},inplace = True)
a_df.head()

,id,away_team,away_short
0,124,Schalke 04,SCH
1,262,Arminia Bielefeld,ARM
2,121,Augsburg,AUG
3,120,Hoffenheim,HOF
4,122,Hertha Berlin,HER


In [10]:
goal_df = pd.json_normalize(matches_df['goals'])
goal_df.rename(columns = {"h":"home_goals","a":"away_goals"},inplace = True)
goal_df.head()

,home_goals,away_goals
0,8,0
1,1,1
2,1,3
3,2,3
4,1,4


In [11]:
xg_df = pd.json_normalize(matches_df["xG"])
xg_df.rename(columns = {"h":"home_xG","a":"away_xG"},inplace = True)
xg_df.head()

,home_xG,away_xG
0,4.62088,0.14975
1,2.46928,0.618787
2,1.04515,1.42046
3,2.40801,2.9032
4,0.495892,2.05406


In [12]:
forecast_df = pd.json_normalize(matches_df["forecast"])
forecast_df.rename(columns = {"w":"win","d":"draw","l":"loss"},inplace = True)
forecast_df.head()

,win,draw,loss
0,0.9948,0.0046,0.0006
1,0.8454,0.1158,0.0388
2,0.243,0.2882,0.4688
3,0.2453,0.253,0.5017
4,0.0616,0.1641,0.7743


In [13]:
matches_df = pd.concat([matches_df[['league', 'year']],h_df,a_df,goal_df,xg_df,forecast_df],axis = 1)
matches_df.head(10)

,league,year,id,home_team,home_short,id,away_team,away_short,home_goals,away_goals,home_xG,away_xG,win,draw,loss
0,bundesliga,2020,117,Bayern Munich,BAY,124,Schalke 04,SCH,8,0,4.62088,0.14975,0.9948,0.0046,0.0006
1,bundesliga,2020,132,Eintracht Frankfurt,EIN,262,Arminia Bielefeld,ARM,1,1,2.46928,0.618787,0.8454,0.1158,0.0388
2,bundesliga,2020,240,Union Berlin,UNI,121,Augsburg,AUG,1,3,1.04515,1.42046,0.243,0.2882,0.4688
3,bundesliga,2020,134,FC Cologne,COL,120,Hoffenheim,HOF,2,3,2.40801,2.9032,0.2453,0.253,0.5017
4,bundesliga,2020,123,Werder Bremen,WER,122,Hertha Berlin,HER,1,4,0.495892,2.05406,0.0616,0.1641,0.7743
5,bundesliga,2020,133,VfB Stuttgart,STU,135,Freiburg,FRE,2,3,2.15741,2.07011,0.3889,0.2421,0.369
6,bundesliga,2020,129,Borussia Dortmund,DOR,130,Borussia M.Gladbach,BMG,3,0,1.70398,0.510293,0.7372,0.2031,0.0597
7,bundesliga,2020,136,RasenBallsport Leipzig,RBL,125,Mainz 05,MAI,3,1,3.22891,1.07201,0.8409,0.1113,0.0478
8,bundesliga,2020,131,Wolfsburg,WOL,119,Bayer Leverkusen,LEV,0,0,1.27652,0.441229,0.6238,0.2774,0.0988
9,bundesliga,2020,122,Hertha Berlin,HER,132,Eintracht Frankfurt,EIN,1,3,1.55994,1.62111,0.3391,0.2951,0.3658


In [14]:
teams_df.head()

,h_a,xG,xGA,npxG,npxGA,ppda,ppda_allowed,deep,deep_allowed,scored,...,date,wins,draws,loses,pts,npxGD,team_id,team_name,league,year
0,h,4.62088,0.149750,3.86311,0.149750,"{'att': 203, 'def': 32}","{'att': 380, 'def': 15}",24,3,8,...,2020-09-18 18:30:00,1,0,0,3,3.713360,117,Bayern Munich,bundesliga,2020
1,a,1.17311,2.523390,1.17311,1.765620,"{'att': 99, 'def': 17}","{'att': 341, 'def': 26}",11,6,1,...,2020-09-27 13:30:00,0,0,1,0,-0.592510,117,Bayern Munich,bundesliga,2020
2,h,3.99311,1.245090,3.23533,1.245090,"{'att': 157, 'def': 38}","{'att': 280, 'def': 23}",11,1,4,...,2020-10-04 16:00:00,1,0,0,3,1.990240,117,Bayern Munich,bundesliga,2020
3,a,3.29447,1.212350,3.29447,1.212350,"{'att': 223, 'def': 30}","{'att': 416, 'def': 12}",16,4,4,...,2020-10-17 16:30:00,1,0,0,3,2.082120,117,Bayern Munich,bundesliga,2020
4,h,3.80823,0.429229,3.80823,0.429229,"{'att': 220, 'def': 22}","{'att': 372, 'def': 22}",12,1,5,...,2020-10-24 13:30:00,1,0,0,3,3.379001,117,Bayern Munich,bundesliga,2020


In [15]:
matches_df['league'].value_counts()

league
la_liga           1900
premier_league    1900
serie_a           1900
ligue_1           1752
bundesliga        1530
Name: count, dtype: int64

In [16]:
team_list=[]

for league in leagues:
    for year in years:
        df1 = pd.read_json(f"../data/raw/understat/{league}/teams_{year}.json")
        team_ids = df1.iloc[0]
        team_names = df1.iloc[1]

        clean_df = pd.DataFrame({'team_id': team_ids,'team_name': team_names}).dropna()
        clean_df['league'] = league
        clean_df['year'] = year
        team_list.append(clean_df)

In [17]:
print(team_list)

[    team_id               team_name      league  year
117     117           Bayern Munich  bundesliga  2020
119     119        Bayer Leverkusen  bundesliga  2020
120     120              Hoffenheim  bundesliga  2020
121     121                Augsburg  bundesliga  2020
122     122           Hertha Berlin  bundesliga  2020
123     123           Werder Bremen  bundesliga  2020
124     124              Schalke 04  bundesliga  2020
125     125                Mainz 05  bundesliga  2020
129     129       Borussia Dortmund  bundesliga  2020
130     130     Borussia M.Gladbach  bundesliga  2020
131     131               Wolfsburg  bundesliga  2020
132     132     Eintracht Frankfurt  bundesliga  2020
133     133           VfB Stuttgart  bundesliga  2020
134     134              FC Cologne  bundesliga  2020
135     135                Freiburg  bundesliga  2020
136     136  RasenBallsport Leipzig  bundesliga  2020
240     240            Union Berlin  bundesliga  2020
262     262       Arminia B

In [18]:
teams_df = pd.concat(team_list,ignore_index = True)

In [19]:
teams_df.head()

,team_id,team_name,league,year
0,117,Bayern Munich,bundesliga,2020
1,119,Bayer Leverkusen,bundesliga,2020
2,120,Hoffenheim,bundesliga,2020
3,121,Augsburg,bundesliga,2020
4,122,Hertha Berlin,bundesliga,2020


In [20]:
players_df.head()

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup,league,year
0,227,Robert Lewandowski,29,2467,41,32.077339,7,4.815550,135,32,4,0,F S,Bayern Munich,33,25.257350,31.740188,5.689343,bundesliga,2020
1,6170,André Silva,32,2787,28,25.599132,5,5.467086,114,31,1,0,F,Eintracht Frankfurt,21,20.294786,26.746589,4.096429,bundesliga,2020
2,8260,Erling Haaland,28,2416,27,23.598484,6,4.035448,92,27,2,0,F S,Borussia Dortmund,25,20.567369,27.197287,5.896189,bundesliga,2020
3,956,Andrej Kramaric,28,2386,20,15.525600,5,4.072423,95,38,2,0,F M S,Hoffenheim,15,11.736730,18.057688,5.372287,bundesliga,2020
4,7052,Wout Weghorst,34,2954,20,18.310400,8,5.427310,93,45,3,0,F S,Wolfsburg,18,16.037062,24.294329,5.954853,bundesliga,2020


In [21]:
matches_df.head()

,league,year,id,home_team,home_short,id,away_team,away_short,home_goals,away_goals,home_xG,away_xG,win,draw,loss
0,bundesliga,2020,117,Bayern Munich,BAY,124,Schalke 04,SCH,8,0,4.62088,0.14975,0.9948,0.0046,0.0006
1,bundesliga,2020,132,Eintracht Frankfurt,EIN,262,Arminia Bielefeld,ARM,1,1,2.46928,0.618787,0.8454,0.1158,0.0388
2,bundesliga,2020,240,Union Berlin,UNI,121,Augsburg,AUG,1,3,1.04515,1.42046,0.243,0.2882,0.4688
3,bundesliga,2020,134,FC Cologne,COL,120,Hoffenheim,HOF,2,3,2.40801,2.9032,0.2453,0.253,0.5017
4,bundesliga,2020,123,Werder Bremen,WER,122,Hertha Berlin,HER,1,4,0.495892,2.05406,0.0616,0.1641,0.7743


In [22]:
matches_df.columns = ['league','year','home_team_id','home_team','home_short','away_team_id','away_team','away_short','home_goals','away_goals','home_xG','away_xG','win','draw','loss']

In [23]:
matches_df.head()

,league,year,home_team_id,home_team,home_short,away_team_id,away_team,away_short,home_goals,away_goals,home_xG,away_xG,win,draw,loss
0,bundesliga,2020,117,Bayern Munich,BAY,124,Schalke 04,SCH,8,0,4.62088,0.14975,0.9948,0.0046,0.0006
1,bundesliga,2020,132,Eintracht Frankfurt,EIN,262,Arminia Bielefeld,ARM,1,1,2.46928,0.618787,0.8454,0.1158,0.0388
2,bundesliga,2020,240,Union Berlin,UNI,121,Augsburg,AUG,1,3,1.04515,1.42046,0.243,0.2882,0.4688
3,bundesliga,2020,134,FC Cologne,COL,120,Hoffenheim,HOF,2,3,2.40801,2.9032,0.2453,0.253,0.5017
4,bundesliga,2020,123,Werder Bremen,WER,122,Hertha Berlin,HER,1,4,0.495892,2.05406,0.0616,0.1641,0.7743


In [24]:
team_list = []
for league in leagues:
    for year in years:
        with open(f"../data/raw/understat/{league}/teams_{year}.json") as f:
            raw = json.load(f)

            for team_id, team_data in raw.items():
                history = team_data['history']
                df = pd.DataFrame(history)

                df['team_id'] = team_id
                df['team_name'] = team_data['title']
                df['league'] = league
                df['year'] = year

                teams_list.append(df)

teams_df = pd.concat(teams_list,ignore_index=True)

In [25]:
teams_df.head()

,h_a,xG,xGA,npxG,npxGA,ppda,ppda_allowed,deep,deep_allowed,scored,...,date,wins,draws,loses,pts,npxGD,team_id,team_name,league,year
0,h,4.62088,0.149750,3.86311,0.149750,"{'att': 203, 'def': 32}","{'att': 380, 'def': 15}",24,3,8,...,2020-09-18 18:30:00,1,0,0,3,3.713360,117,Bayern Munich,bundesliga,2020
1,a,1.17311,2.523390,1.17311,1.765620,"{'att': 99, 'def': 17}","{'att': 341, 'def': 26}",11,6,1,...,2020-09-27 13:30:00,0,0,1,0,-0.592510,117,Bayern Munich,bundesliga,2020
2,h,3.99311,1.245090,3.23533,1.245090,"{'att': 157, 'def': 38}","{'att': 280, 'def': 23}",11,1,4,...,2020-10-04 16:00:00,1,0,0,3,1.990240,117,Bayern Munich,bundesliga,2020
3,a,3.29447,1.212350,3.29447,1.212350,"{'att': 223, 'def': 30}","{'att': 416, 'def': 12}",16,4,4,...,2020-10-17 16:30:00,1,0,0,3,2.082120,117,Bayern Munich,bundesliga,2020
4,h,3.80823,0.429229,3.80823,0.429229,"{'att': 220, 'def': 22}","{'att': 372, 'def': 22}",12,1,5,...,2020-10-24 13:30:00,1,0,0,3,3.379001,117,Bayern Munich,bundesliga,2020


In [26]:
teams_df.shape

(35928, 23)

In [27]:
# derived columns for matches
matches_df['total_goals'] = matches_df['home_goals'] + matches_df['away_goals']
matches_df['total_xG']    = matches_df['home_xG'] + matches_df['away_xG']

# save all three
import os
os.makedirs("../data/processed", exist_ok=True)

matches_df.to_csv("../data/processed/matches.csv", index=False)
players_df.to_csv("../data/processed/players.csv", index=False)
teams_df.to_csv("../data/processed/teams.csv",     index=False)

print("Saved successfully")
print("matches_df:", matches_df.shape)
print("teams_df:  ", teams_df.shape)
print("players_df:", players_df.shape)

Saved successfully
matches_df: (8982, 17)
teams_df:   (35928, 23)
players_df: (13964, 20)
